# 05 · Revision experiments (Array, ARRAY-D-26-07236)

New analyses requested by the reviewers. Notebooks `01`–`04` are unchanged and remain the code behind the
submitted numbers; everything here is additional. Data loading, map construction, the model and the training
settings are copied from `01` (held-out split) and `02` (cross-validation).

| Part | Question | What runs |
|---|---|---|
| `stats` | Dataset statistics (Reviewer 2) | trial counts, epoch length, class order |
| `E1` | Run-to-run variance (single seed) | SD, pooled, SI on the held-out split, several seeded runs |
| `E2` | Pooling hurts because the model is small? Alignment? | pooled and SI with ~4× / ~13× wider models and Euclidean alignment |
| `E3` | Pooled − SI conflates seeing the subject with having more data? | pooled on 14 subjects including the target (same size as SI) |
| `E4` | Blocked folds are not leakage-safe by themselves | SD 5-fold CV with instruction-block groups, plus shuffled and blocked (seeded) |
| `E5` | Temporal drift / adjacency | class from trial position only; class from slow (mean, slope) features |
| `E6` | Permutation null | SD and pooled trained on permuted labels |
| `E7` | Ocular / muscle / slow-drift artefacts | SD and pooled with peripheral electrodes removed and/or 1 Hz high-pass |
| `E8` | Matched baseline | EEGNet-8,2 under SD, pooled and SI on the same split |

**How to run on Kaggle.** Attach the BCIC2020-3 dataset (same as `01`), accelerator GPU T4. Optionally attach
the output of `01` (its `paper/maps.pkl` is reused, which saves the map build). Run with *Save Version → Save &
Run All (Commit)*. Every finished model is checkpointed to `/kaggle/working/revision/`, and no new model is
started after `TIME_BUDGET_H`. To continue in a new session, attach the previous version's output as an input
and run again: finished runs are skipped. The last cell writes `revision_results.zip` (tables, per-run metrics,
predictions; no map caches).

**Why a script.** The analysis code is written to `revision_experiments.py` (next cells) and each part runs in
its own Python process, restarted every 40 models. TensorFlow/Keras keeps memory for every model built, even after
`clear_session()`; over the ~900 models of this notebook that exhausted the session's memory in a first run. XLA
compilation is turned off for the same reason (it compiled every new model). With two GPUs, two parts run at the
same time, one per GPU.

In [ ]:
# ---------------------------------------------------------------- run settings (edit here)
import os, sys, json, time, subprocess
RUN = ['stats','E5','E1','E3','E6','E4','E7','E2','E8','E4b']   # parts, in priority order; remove any to skip
SETTINGS = dict(TIME_BUDGET_H=10.5,                    # no new model after this (Kaggle sessions end at 12 h)
                SEEDS_SD=[0,1,2,3,4], SEEDS_POOLED=[0,1,2,3,4], SEEDS_SI=[0,1,2],
                N_PERM_SD=10, N_PERM_POOLED=5,
                MAX_FITS_PER_PROCESS=40)               # a fresh process every 40 models keeps memory bounded
json.dump(SETTINGS, open('rev05_settings.json','w'))

## Analysis code

Data loading and maps (`make_maps` is `build_maps` from `01`, with optional Euclidean alignment, 1 Hz high-pass
and electrode subset, all off for the `base` variant), the model (`build_model` from `01`, widths as arguments,
seeded initialisation), the training settings of `01` (held-out split) and `02` (cross-validation), the regime
runners, EEGNet-8,2, the drift controls, and the summary tables.

In [ ]:
%%writefile revision_experiments.py
"""Revision experiments for ARRAY-D-26-07236, written by notebooks/05_revision_experiments.ipynb.
Usage: python revision_experiments.py <stats|E1|...|summary>. Data loading, maps, model and training
settings are those of notebooks 01 (held-out split) and 02 (cross-validation)."""
import os, sys
import numpy as np, scipy.io, glob, time, gc, json, pickle, shutil, zipfile
import pandas as pd
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
from scipy.signal import butter, sosfiltfilt, decimate
from scipy import stats
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, f1_score, accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, regularizers, constraints
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g,True)
tf.get_logger().setLevel('ERROR')
print('TF',tf.__version__,'| GPUs',len(tf.config.list_physical_devices('GPU')))

# ---------------------------------------------------------------- run settings (from rev05_settings.json)
SETTINGS=json.load(open('rev05_settings.json')) if os.path.exists('rev05_settings.json') else {}
TIME_BUDGET_H=SETTINGS.get('TIME_BUDGET_H',10.5)
MAX_FITS=SETTINGS.get('MAX_FITS_PER_PROCESS',40)   # fresh process after this many models (bounds memory)
SEEDS_SD,SEEDS_POOLED,SEEDS_SI=SETTINGS.get('SEEDS_SD',[0,1,2,3,4]),SETTINGS.get('SEEDS_POOLED',[0,1,2,3,4]),SETTINGS.get('SEEDS_SI',[0,1,2])
N_PERM_SD,N_PERM_POOLED=SETTINGS.get('N_PERM_SD',10),SETTINGS.get('N_PERM_POOLED',5)
SMOKE = os.environ.get('SMOKE') == '1'   # tiny local test only
if SMOKE:
    SEEDS_SD, SEEDS_POOLED, SEEDS_SI = [0,1], [0], [0]; N_PERM_SD, N_PERM_POOLED = 2, 1
N_EVAL = 3 if SMOKE else None   # smoke test: evaluate only the first 3 subjects in per-subject loops

# ---------------------------------------------------------------- config copied from 01 (LOCKED)
NUM_CLASSES=5; FS=256; MAP=32; WSTRIDE=3; KFOLD=5
CFG=dict(l2=1e-4, sd=0.2, do=0.4)
classNames=['Hello','Help me','Stop','Thank you','Yes']
label_to_channel={
    'Fp1':0,'Fp2':1,'F7':2,'F3':3,'Fz':4,'F4':5,'F8':6,'FC5':7,'FC1':8,'FC2':9,'FC6':10,
    'T7':11,'C3':12,'Cz':13,'C4':14,'T8':15,'TP9':16,'CP5':17,'CP1':18,'CP2':19,'CP6':20,
    'TP10':21,'P7':22,'P3':23,'Pz':24,'P4':25,'P8':26,'PO9':27,'O1':28,'Oz':29,'O2':30,
    'PO10':31,'AF7':32,'AF3':33,'AF4':34,'AF8':35,'F5':36,'F1':37,'F2':38,'F6':39,'FT9':40,
    'FT7':41,'FC3':42,'FC4':43,'FT8':44,'FT10':45,'C5':46,'C1':47,'C2':48,'C6':49,'TP7':50,
    'CP3':51,'CPz':52,'CP4':53,'TP8':54,'P5':55,'P1':56,'P2':57,'P6':58,'PO7':59,'PO3':60,
    'POz':61,'PO4':62,'PO8':63}
selected=list(label_to_channel.keys()); sel_idx=np.array([label_to_channel[c] for c in selected])
# E7: electrodes most exposed to ocular (frontopolar/anterior) and muscle (temporal) activity
PERIPHERAL=['Fp1','Fp2','AF7','AF3','AF4','AF8','F7','F8','FT9','FT10','FT7','FT8',
            'T7','T8','TP9','TP10','TP7','TP8','PO9','PO10']
CENTRAL=[c for c in selected if c not in PERIPHERAL]

# ---------------------------------------------------------------- paths (same detection as 01)
_ROOT=os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())=='notebooks' else os.getcwd()
ON_KAGGLE=os.path.exists('/kaggle/working')
RESULTS_DIR=os.environ.get('RESULTS_DIR') or ('/kaggle/working' if ON_KAGGLE else os.path.join(_ROOT,'results'))
DATA_ROOT=os.environ.get('BCIC_DATA')
if DATA_ROOT is None:
    for path in ['/kaggle/input/datasets/sanaephd/imu-shah-2024-dataset',
                 '/kaggle/input/imu-shah-2024-dataset','/kaggle/input/imu-shah-2024']+\
                ([f'/kaggle/input/{d}' for d in os.listdir('/kaggle/input')
                  if ('imu' in d.lower() or 'shah' in d.lower())] if os.path.exists('/kaggle/input') else [])+\
                [os.path.join(_ROOT,'data')]:
        if os.path.exists(os.path.join(path,'Train')) and os.path.exists(os.path.join(path,'valid')):
            DATA_ROOT=path; break
train_files=sorted(glob.glob(os.path.join(DATA_ROOT,'Train','*.mat')))
valid_files=sorted(glob.glob(os.path.join(DATA_ROOT,'valid','*.mat')))
OUT=os.path.join(RESULTS_DIR,'revision'); os.makedirs(OUT,exist_ok=True)
RUNS=f'{OUT}/runs.csv'; PREDS=f'{OUT}/preds.pkl'
print('dataset:',DATA_ROOT,'|',len(train_files),'subjects | output:',OUT)

# resume: copy checkpoints (runs*.csv, preds*.pkl, maps caches, E5) from an attached earlier output, once
if os.path.exists('/kaggle/input') and not glob.glob(f'{OUT}/runs*.csv'):
    found=sorted({os.path.dirname(f) for f in glob.glob('/kaggle/input/**/runs*.csv',recursive=True)
                  if glob.glob(os.path.join(os.path.dirname(f),'preds*.pkl'))})
    for d in found:
        for f in glob.glob(f'{d}/*'):
            name=os.path.basename(f)
            if name.startswith(('runs','preds','maps_','E5_')) and not os.path.exists(f'{OUT}/{name}'):
                shutil.copy(f,f'{OUT}/{name}')
        print('resumed from',d)
PAPER_MAPS=next(iter(glob.glob('/kaggle/input/**/paper/maps.pkl',recursive=True)),None) if os.path.exists('/kaggle/input') else None
if PAPER_MAPS is None and os.path.exists(os.path.join(RESULTS_DIR,'paper','maps.pkl')): PAPER_MAPS=os.path.join(RESULTS_DIR,'paper','maps.pkl')

T0=time.time()
DEADLINE=float(os.environ.get('REV05_DEADLINE', T0+TIME_BUDGET_H*3600))   # set by the notebook
class OutOfTime(Exception): pass
def check_time():
    if time.time() > DEADLINE: raise OutOfTime

def _todict(o):
    d={}
    for f in o._fieldnames:
        e=o.__dict__[f]; d[f]=_todict(e) if isinstance(e,scipy.io.matlab.mat_struct) else e
    return d
def _check(d):
    for k in d:
        if isinstance(d[k],scipy.io.matlab.mat_struct): d[k]=_todict(d[k])
    return d
def load_epo(file):
    md=_check(scipy.io.loadmat(file,struct_as_record=False,squeeze_me=True))
    epo=next(md[k] for k in ['epo_train','epo_test','epo_validation','epo'] if k in md)
    return epo['x'], np.argmax(epo['y'],axis=0)          # x: time x channels x trials

mnt0=_check(scipy.io.loadmat(train_files[0],struct_as_record=False,squeeze_me=True))['mnt']
x_sel,y_sel=mnt0['x'][sel_idx],mnt0['y'][sel_idx]
xi,yi=np.meshgrid(np.linspace(x_sel.min(),x_sel.max(),MAP),np.linspace(y_sel.min(),y_sel.max(),MAP))
tree=cKDTree(np.c_[x_sel,y_sel]); d1,_=tree.query(np.c_[xi.ravel(),yi.ravel()],k=1)
dd,_=tree.query(np.c_[x_sel,y_sel],k=2); MASK=(d1.reshape(MAP,MAP)<=1.2*np.median(dd[:,1]))

RAW=[]   # per subject: raw train / valid epochs, as stored
for i,(tf_,vf_) in enumerate(zip(train_files,valid_files)):
    xa,ya=load_epo(tf_); xb,yb=load_epo(vf_)
    RAW.append(dict(id=i,name=os.path.basename(tf_).replace('.mat',''),xtr=xa,ytr=ya,xte=xb,yte=yb))
print('loaded raw epochs:',len(RAW),'subjects | train',RAW[0]['xtr'].shape,'| valid',RAW[0]['xte'].shape)

def window_indices(total, win_ms=125, overlap=50, start_map=6, end_map=44, wstride=WSTRIDE):
    win=round(FS*win_ms/1000); ov=round(win*overlap/100); step=win-ov
    n_maps=int(np.floor((total-ov)/step))
    wins=[w for w in range(n_maps) if (start_map-1<=w<end_map) and ((w-(start_map-1))%wstride==0)]
    return [(w*step, w*step+win) for w in wins]

def make_maps(data, keep=None):
    """build_maps from 01 on an epoch array (time x channels x trials); keep = electrodes used."""
    idx=sel_idx if keep is None else np.array([label_to_channel[c] for c in keep])
    xs,ys=mnt0['x'][idx],mnt0['y'][idx]
    filt=data.copy() if keep is None else data[:,idx,:].copy()
    n_trials=filt.shape[2]
    for t in range(n_trials):
        tr=filt[:,:,t]; filt[:,:,t]=tr-tr.mean(axis=1,keepdims=True)
    W=window_indices(data.shape[0])
    out=np.zeros((n_trials,len(W),MAP,MAP,1),dtype=np.float32)
    for t in range(n_trials):
        dt=filt[:,:,t] if keep is not None else filt[:,sel_idx,t]
        for wi,(s,e) in enumerate(W):
            v=dt[s:e,:].mean(0)
            zi=griddata((xs,ys),v,(xi,yi),method='cubic'); zi=np.where(MASK,zi,np.nan)
            if np.isfinite(zi).any():
                lo,hi=np.nanmin(zi),np.nanmax(zi); zn=(zi-lo)/(hi-lo) if hi>lo else np.zeros_like(zi)
            else: zn=np.zeros_like(zi)
            zn=np.nan_to_num(zn); zn[~MASK]=0.0; out[t,wi,:,:,0]=zn
    return out

def highpass(data, fc=1.0, order=4):
    sos=butter(order, fc, btype='highpass', fs=FS, output='sos')
    return sosfiltfilt(sos, data, axis=0)

def ea_matrix(x):
    """Euclidean alignment (He & Wu, 2020): R^-1/2 of the mean trial covariance (unlabelled trials)."""
    C=np.mean([x[:,:,t].T@x[:,:,t]/x.shape[0] for t in range(x.shape[2])],axis=0)
    C=C+1e-6*np.trace(C)/C.shape[0]*np.eye(C.shape[0])
    w,V=np.linalg.eigh(C); return (V*(1/np.sqrt(np.maximum(w,1e-12*w.max()))))@V.T
def apply_ea(x,P): return np.einsum('tcn,dc->tdn',x,P)

def variant_data(s, variant):
    xtr,xte=s['xtr'],s['xte']
    if 'ea' in variant.split('_'):
        P=ea_matrix(xtr)   # each subject aligned with its own training-partition trials, labels unused
        xtr,xte=apply_ea(xtr,P),apply_ea(xte,P)
    if 'hp1' in variant.split('_'): xtr,xte=highpass(xtr),highpass(xte)
    return xtr,xte

MAPS={}
def get_maps(variant):
    """variant: 'base' | 'ea' | 'hp1' | 'central' | 'hp1_central' ; cached to disk."""
    if variant in MAPS: return MAPS[variant]
    cache=f'{OUT}/maps_{variant}.pkl'
    if os.path.exists(cache):
        with open(cache,'rb') as f: MAPS[variant]=pickle.load(f); return MAPS[variant]
    if variant=='base' and PAPER_MAPS:
        with open(PAPER_MAPS,'rb') as f: MAPS['base']=pickle.load(f)
        print('base maps: paper cache',PAPER_MAPS); return MAPS['base']
    keep=CENTRAL if 'central' in variant.split('_') else None
    t0=time.time(); subj=[]
    for s in RAW:
        xtr,xte=variant_data(s,variant)
        subj.append(dict(id=s['id'],name=s['name'],Xtr=make_maps(xtr,keep),ytr=s['ytr'],Xte=make_maps(xte,keep),yte=s['yte']))
    with open(cache,'wb') as f: pickle.dump(subj,f)
    print(f'maps[{variant}] built in {time.time()-t0:.0f}s'); MAPS[variant]=subj; return subj

base=get_maps('base'); INP=base[0]['Xtr'].shape[1:]; print('input',INP)

def build_model(input_shape=None,n_classes=NUM_CLASSES,w1=32,w2=64,wb=16,l2=CFG['l2'],sd=CFG['sd'],do=CFG['do'],lr=1e-3):
    input_shape=input_shape or INP
    reg=regularizers.l2(l2); inp=layers.Input(shape=input_shape)
    x=layers.TimeDistributed(layers.Conv2D(w1,3,padding='same',activation='relu',kernel_regularizer=reg))(inp)
    x=layers.TimeDistributed(layers.BatchNormalization())(x)
    x=layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x=layers.TimeDistributed(layers.SpatialDropout2D(sd))(x)
    x=layers.TimeDistributed(layers.Conv2D(w2,3,padding='same',activation='relu',kernel_regularizer=reg))(x)
    x=layers.TimeDistributed(layers.BatchNormalization())(x)
    x=layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x=layers.TimeDistributed(layers.SpatialDropout2D(sd))(x)
    x=layers.TimeDistributed(layers.Conv2D(wb,1,padding='same',activation='relu',kernel_regularizer=reg))(x)
    x=layers.TimeDistributed(layers.Flatten())(x)
    x=layers.GlobalAveragePooling1D()(x)
    x=layers.Dropout(do)(x)
    out=layers.Dense(n_classes,activation='softmax',kernel_regularizer=reg)(x)
    m=models.Model(inp,out); m.compile(optimizer=optimizers.Adam(lr),loss='categorical_crossentropy',metrics=['accuracy'],jit_compile=False)
    return m
WIDTHS={'w25k':dict(w1=32,w2=64,wb=16),'w100k':dict(w1=64,w2=128,wb=32),'w330k':dict(w1=128,w2=256,wb=64)}

def strat_val(y,val_frac=0.15,seed=0):                    # 01
    rng=np.random.RandomState(seed); tr=[]; va=[]
    for c in range(NUM_CLASSES):
        ci=np.where(y==c)[0]; rng.shuffle(ci); k=max(1,int(len(ci)*val_frac)); va+=list(ci[:k]); tr+=list(ci[k:])
    return np.array(tr),np.array(va)
def inner_val(y_tr_idx, y, seed=0):                       # 02
    rng=np.random.RandomState(seed); tr=[]; va=[]
    for c in range(NUM_CLASSES):
        ci=y_tr_idx[y[y_tr_idx]==c]; rng.shuffle(ci); k=max(1,int(len(ci)*0.15)); va+=list(ci[:k]); tr+=list(ci[k:])
    return np.array(tr),np.array(va)

EPOCHS_HO, EPOCHS_CV = (2,2) if SMOKE else (80,70)
N_FITS=0
class Restart(Exception): pass   # this process has trained MAX_FITS models: exit and let the notebook start a fresh one
def unit_start():
    check_time()
    if N_FITS>=MAX_FITS: raise Restart
def _train(m,X,y,tr,va,epochs,pat_es,pat_lr):
    global N_FITS; N_FITS+=1
    m.fit(X[tr],to_categorical(y[tr],NUM_CLASSES),validation_data=(X[va],to_categorical(y[va],NUM_CLASSES)),
          epochs=epochs,batch_size=32,verbose=0,
          callbacks=[EarlyStopping('val_loss',patience=pat_es,restore_best_weights=True,verbose=0),
                     ReduceLROnPlateau('val_loss',factor=0.5,patience=pat_lr,min_lr=1e-6,verbose=0)])
    return m
def fit(X,y,split_seed,init_seed,builder=None,**kw):       # held-out split: 01 settings
    check_time(); tr,va=strat_val(y,0.15,split_seed)
    keras.utils.set_random_seed(init_seed); m=(builder or build_model)(**kw)
    return _train(m,X,y,tr,va,EPOCHS_HO,15,6)
def fit_fold(X,y,tr_idx,split_seed,init_seed,builder=None,**kw):   # cross-validation: 02 settings
    check_time(); tr,va=inner_val(tr_idx,y,split_seed)
    keras.utils.set_random_seed(init_seed); m=(builder or build_model)(**kw)
    return _train(m,X,y,tr,va,EPOCHS_CV,12,5)
def predict(m,X):   # same outputs as m.predict (inference mode), without building a predict function per model
    return np.argmax(np.concatenate([m(X[i:i+256],training=False).numpy() for i in range(0,len(X),256)]),axis=1)
def free(*objs):
    keras.backend.clear_session(); gc.collect()

# ---- fold constructions (02) + instruction-block groups (new)
def strat_shuffled_folds(y,k=KFOLD,seed=0):
    rng=np.random.RandomState(seed); folds=[[] for _ in range(k)]
    for c in range(NUM_CLASSES):
        idx=np.where(y==c)[0]; rng.shuffle(idx)
        for i,ix in enumerate(idx): folds[i%k].append(int(ix))
    return [np.array(sorted(f)) for f in folds]
def strat_blocked_folds(y, k=KFOLD, seed=0):
    folds=[[] for _ in range(k)]
    for c in range(NUM_CLASSES):
        idx=np.where(y==c)[0]
        for i,s in enumerate(np.array_split(idx,k)): folds[i].extend(s.tolist())
    return [np.array(sorted(f)) for f in folds]
def block_groups(y, max_len=4):
    """Runs of consecutive same-class trials in storage order; runs longer than max_len (class-sorted
    storage) are cut into consecutive chunks of max_len, the instruction-block size."""
    g=np.zeros(len(y),int); gid=0; start=0
    for i in range(1,len(y)+1):
        if i==len(y) or y[i]!=y[i-1]:
            for j in range(start,i): g[j]=gid+(j-start)//max_len
            gid+=(i-start-1)//max_len+1; start=i
    return g
def group_folds(y,k=KFOLD,seed=0):
    sgk=StratifiedGroupKFold(n_splits=k,shuffle=True,random_state=seed)
    return [np.sort(te) for _,te in sgk.split(np.zeros(len(y)),y,block_groups(y))]
FOLDS={'shuffled':strat_shuffled_folds,'blocked':strat_blocked_folds,'group':group_folds}

# ---- bookkeeping: one row per (experiment, variant, model, regime, seed, subject)
# Each part writes its own runs_<part>.csv / preds_<part>.pkl (two parts can run at once, one per GPU);
# all files present are read to skip finished runs. Writes are atomic (temporary file, then rename).
COLS=['exp','variant','model','regime','seed','subject','bal_acc','kappa','wf1','acc','n_train','n_test','secs']
KEYCOLS=['exp','variant','model','regime','seed','subject']
PART=sys.argv[1] if len(sys.argv)>1 else 'nb'
RUNS_OWN=f'{OUT}/runs_{PART}.csv'; PREDS_OWN=f'{OUT}/preds_{PART}.pkl'
def _atomic(path,write):
    tmp=f'{path}.tmp{os.getpid()}'; write(tmp); os.replace(tmp,path)
def load_all():
    frames=[]
    for f in sorted(glob.glob(f'{OUT}/runs*.csv')):
        try: frames.append(pd.read_csv(f,on_bad_lines='skip'))   # a file cut by a killed process loses at most its last line
        except Exception as e: print('skipped unreadable',f,e)
    R=pd.concat(frames,ignore_index=True).drop_duplicates(KEYCOLS,keep='last') if frames else pd.DataFrame(columns=COLS)
    P={}
    for f in sorted(glob.glob(f'{OUT}/preds*.pkl')):
        try: P.update(pickle.load(open(f,'rb')))
        except Exception as e: print('skipped unreadable',f,e)
    return R,P
R,P=load_all()
ROWS_OWN=pd.read_csv(RUNS_OWN).to_dict('records') if os.path.exists(RUNS_OWN) else []
P_OWN=pickle.load(open(PREDS_OWN,'rb')) if os.path.exists(PREDS_OWN) else {}
def key(exp,variant,model,regime,seed,sid): return f'{exp}|{variant}|{model}|{regime}|{seed}|{sid}'
def is_done(*k): return key(*k) in P
def record(k,yt,yp,n_train,secs):
    exp,variant,model,regime,seed,sid=k
    row=dict(exp=exp,variant=variant,model=model,regime=regime,seed=seed,subject=RAW[sid]['name'],
             bal_acc=balanced_accuracy_score(yt,yp)*100,kappa=cohen_kappa_score(yt,yp),
             wf1=f1_score(yt,yp,average='weighted')*100,acc=accuracy_score(yt,yp)*100,
             n_train=n_train,n_test=len(yt),secs=round(secs,1))
    ROWS_OWN.append(row); P_OWN[key(*k)]=P[key(*k)]=(np.asarray(yt,np.int8),np.asarray(yp,np.int8))
    _atomic(RUNS_OWN,lambda f: pd.DataFrame(ROWS_OWN,columns=COLS).to_csv(f,index=False))
    _atomic(PREDS_OWN,lambda f: pickle.dump(P_OWN,open(f,'wb')))
def subj_mean(exp,regime,variant=None,model=None):
    q=R[(R.exp==exp)&(R.regime==regime)]
    if variant is not None: q=q[q.variant==variant]
    if model is not None: q=q[q.model==model]
    return q

def run_sd(exp,variant,seed,model='w25k',perm=None):
    S=get_maps(variant)
    for s in S[:N_EVAL]:
        k=(exp,variant,model,'sd',seed,s['id'])
        if is_done(*k): continue
        unit_start(); t0=time.time(); y=s['ytr']
        if perm is not None: y=np.random.RandomState(10000+perm*100+s['id']).permutation(y)
        m=fit(s['Xtr'],y,split_seed=s['id'],init_seed=1000*seed+s['id'],**WIDTHS[model])
        record(k,s['yte'],predict(m,s['Xte']),len(y),time.time()-t0); del m; free()

def _pool(S,ids,perm=None):
    X=np.concatenate([S[i]['Xtr'] for i in ids]); ys=[]
    for i in ids:
        y=S[i]['ytr']
        if perm is not None: y=np.random.RandomState(20000+perm*100+i).permutation(y)   # within subject
        ys.append(y)
    return X,np.concatenate(ys)

def run_pooled(exp,variant,seed,model='w25k',perm=None):
    S=get_maps(variant)
    if all(is_done(exp,variant,model,'pooled',seed,s['id']) for s in S): return
    unit_start(); t0=time.time(); X,y=_pool(S,range(len(S)),perm)
    m=fit(X,y,split_seed=7,init_seed=1000*seed+7,**WIDTHS[model]); del X
    for s in S: record((exp,variant,model,'pooled',seed,s['id']),s['yte'],predict(m,s['Xte']),len(y),time.time()-t0)
    del m; free()

def run_si(exp,variant,seed,model='w25k'):
    S=get_maps(variant)
    for s in S[:N_EVAL]:
        k=(exp,variant,model,'si',seed,s['id'])
        if is_done(*k): continue
        unit_start(); t0=time.time(); X,y=_pool(S,[o['id'] for o in S if o['id']!=s['id']])
        m=fit(X,y,split_seed=100+s['id'],init_seed=1000*seed+100+s['id'],**WIDTHS[model]); del X
        record(k,s['yte'],predict(m,s['Xte']),len(y),time.time()-t0); del m; free()

def run_pooled_eq(exp,variant,seed,model='w25k'):
    """E3: target + 13 other subjects = 14 training subjects, the same number as SI for that target."""
    S=get_maps(variant); n=len(S)
    for s in S[:N_EVAL]:
        k=(exp,variant,model,'pooled_eq',seed,s['id'])
        if is_done(*k): continue
        unit_start(); drop=(s['id']+1)%n; t0=time.time(); X,y=_pool(S,[o['id'] for o in S if o['id']!=drop])
        m=fit(X,y,split_seed=200+s['id'],init_seed=1000*seed+200+s['id'],**WIDTHS[model]); del X
        record(k,s['yte'],predict(m,s['Xte']),len(y),time.time()-t0); del m; free()

def run_cv(exp,variant,scheme,seed,model='w25k'):
    """SD 5-fold CV over all 350 trials (train + valid concatenated, as in 02)."""
    S=get_maps(variant)
    for s in S[:N_EVAL]:
        k=(exp,variant,model,f'sd_cv_{scheme}',seed,s['id'])
        if is_done(*k): continue
        unit_start(); X=np.concatenate([s['Xtr'],s['Xte']]); y=np.concatenate([s['ytr'],s['yte']])
        yp=-np.ones(len(y),int); t0=time.time()
        for fi,te in enumerate(FOLDS[scheme](y,seed=1000*seed+s['id'])):
            tr=np.setdiff1d(np.arange(len(y)),te)
            m=fit_fold(X,y,tr,split_seed=1000*fi+s['id'],init_seed=1000*seed+10*fi+s['id'],**WIDTHS[model])
            yp[te]=predict(m,X[te]); del m; free()
        record(k,y,yp,int(len(y)*(KFOLD-1)/KFOLD),time.time()-t0)

def eegnet_input(x):
    W=window_indices(x.shape[0]); a,b=W[0][0],W[-1][1]
    x=x-x.mean(axis=1,keepdims=True)                       # CAR
    x=decimate(x[a:b],2,axis=0,zero_phase=True)            # 256 -> 128 Hz
    return np.transpose(x,(2,1,0))[...,None].astype(np.float32)   # trials x channels x time x 1
def build_eegnet(input_shape,n_classes=NUM_CLASSES,F1=8,D=2,F2=16,kL=64,drop=0.5):
    C,T=input_shape[0],input_shape[1]; inp=layers.Input(shape=input_shape)
    x=layers.Conv2D(F1,(1,kL),padding='same',use_bias=False)(inp); x=layers.BatchNormalization()(x)
    x=layers.DepthwiseConv2D((C,1),use_bias=False,depth_multiplier=D,depthwise_constraint=constraints.max_norm(1.))(x)
    x=layers.BatchNormalization()(x); x=layers.Activation('elu')(x)
    x=layers.AveragePooling2D((1,4))(x); x=layers.Dropout(drop)(x)
    x=layers.SeparableConv2D(F2,(1,16),padding='same',use_bias=False)(x)
    x=layers.BatchNormalization()(x); x=layers.Activation('elu')(x)
    x=layers.AveragePooling2D((1,8))(x); x=layers.Dropout(drop)(x)
    x=layers.Flatten()(x)
    out=layers.Dense(n_classes,activation='softmax',kernel_constraint=constraints.max_norm(0.25))(x)
    m=models.Model(inp,out); m.compile(optimizer=optimizers.Adam(1e-3),loss='categorical_crossentropy',metrics=['accuracy'],jit_compile=False)
    return m
EEG=None
def eeg_data():
    global EEG
    if EEG is None:
        EEG=[dict(id=s['id'],Xtr=eegnet_input(s['xtr']),ytr=s['ytr'],Xte=eegnet_input(s['xte']),yte=s['yte']) for s in RAW]
    return EEG
def run_eegnet(regime,seed):
    E=eeg_data(); n=len(E); shp=E[0]['Xtr'].shape[1:]
    def scaled(Xtr,*Xs):
        sc=Xtr.std(); return [Xtr/sc]+[X/sc for X in Xs]
    if regime=='pooled':
        if all(is_done('E8','raw','eegnet','pooled',seed,e['id']) for e in E): return
        unit_start(); t0=time.time(); X=np.concatenate([e['Xtr'] for e in E]); y=np.concatenate([e['ytr'] for e in E])
        Xs=scaled(X,*[e['Xte'] for e in E])
        m=fit(Xs[0],y,split_seed=7,init_seed=1000*seed+7,builder=build_eegnet,input_shape=shp,drop=0.25)
        for e,Xte in zip(E,Xs[1:]): record(('E8','raw','eegnet','pooled',seed,e['id']),e['yte'],predict(m,Xte),len(y),time.time()-t0)
        del m; free(); return
    for e in E[:N_EVAL]:
        k=('E8','raw','eegnet',regime,seed,e['id'])
        if is_done(*k): continue
        unit_start(); t0=time.time()
        if regime=='sd':
            X,y=e['Xtr'],e['ytr']; ss,init,dr=e['id'],1000*seed+e['id'],0.5
        else:
            ids=[o['id'] for o in E if o['id']!=e['id']]
            X=np.concatenate([E[i]['Xtr'] for i in ids]); y=np.concatenate([E[i]['ytr'] for i in ids])
            ss,init,dr=100+e['id'],1000*seed+100+e['id'],0.25
        Xtr,Xte=scaled(X,e['Xte'])
        m=fit(Xtr,y,split_seed=ss,init_seed=init,builder=build_eegnet,input_shape=shp,drop=dr)
        record(k,e['yte'],predict(m,Xte),len(y),time.time()-t0); del m; free()

def run_e5():
    rows=[]; W=window_indices(RAW[0]['xtr'].shape[0]); a,b=W[0][0],W[-1][1]; tt=np.arange(b-a)-(b-a-1)/2
    for s in RAW:
        x=np.concatenate([s['xtr'],s['xte']],axis=2); y=np.concatenate([s['ytr'],s['yte']])
        x=x-x.mean(axis=1,keepdims=True); seg=x[a:b]
        F=np.concatenate([seg.mean(0).T, (np.tensordot(tt,seg,axes=(0,0))/np.sum(tt**2)).T],axis=1)  # trials x 2C
        pos=np.arange(len(y)).reshape(-1,1)
        runs=np.diff(np.flatnonzero(np.r_[True,y[1:]!=y[:-1],True]))
        r=dict(subject=s['name'],median_run=float(np.median(runs)),max_run=int(runs.max()),n_runs=len(runs))
        for scheme,ff in FOLDS.items():
            for feat,clf,Xf in [('position',KNeighborsClassifier(1),pos),
                                ('slow',LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'),F)]:
                yp=np.zeros(len(y),int)
                for te in ff(y,seed=s['id']):
                    tr=np.setdiff1d(np.arange(len(y)),te); yp[te]=clf.fit(Xf[tr],y[tr]).predict(Xf[te])
                r[f'{feat}_{scheme}']=balanced_accuracy_score(y,yp)*100
        rows.append(r)
    D=pd.DataFrame(rows); D.to_csv(f'{OUT}/E5_drift_controls.csv',index=False); return D

PARTS={   # ordered by priority; finished runs are skipped when the notebook is re-run
 'E1': lambda: ([run_sd('E1','base',sd) for sd in SEEDS_SD],[run_pooled('E1','base',sd) for sd in SEEDS_POOLED],
                [run_si('E1','base',sd) for sd in SEEDS_SI]),
 'E3': lambda: [run_pooled_eq('E3','base',sd) for sd in SEEDS_SI[:1]],
 'E6': lambda: ([run_sd('E6','base',p,perm=p) for p in range(N_PERM_SD)],
                [run_pooled('E6','base',p,perm=p) for p in range(N_PERM_POOLED)]),
 'E4': lambda: run_cv('E4','base','group',0),
 'E7': lambda: ([(run_sd('E7',v,sd),run_pooled('E7',v,sd)) for sd in [0,1,2] for v in ['hp1_central','hp1','central']],
                run_cv('E7','hp1_central','group',0)),
 'E2': lambda: ([run_pooled('E2',v,sd,model=mw) for sd in [0,1] for v,mw in [('base','w330k'),('ea','w25k'),('base','w100k'),('ea','w330k')]],
                [run_si('E2',v,0,model=mw) for v,mw in [('ea','w25k'),('base','w330k')]]),
 'E8': lambda: [run_eegnet(reg,sd) for sd in [0,1,2] for reg in ['sd','pooled','si']],
 'E4b': lambda: [run_cv('E4','base',sch,0) for sch in ['blocked','shuffled']],   # seeded re-runs of the paper's two schemes
}


def summary():
    def per_run(q):            # subject mean for each run
        return q.groupby('seed').bal_acc.mean()
    def summ(q):
        pr=per_run(q); n=q.groupby('seed').size()
        return dict(runs=len(pr), subjects=int(n.min()) if len(n) else 0, mean=pr.mean(), sd_runs=pr.std(ddof=1) if len(pr)>1 else np.nan,
                    sd_subjects=q.groupby('subject').bal_acc.mean().std(ddof=1), kappa=q.kappa.mean(), wf1=q.wf1.mean())
    def subj_avg(q): return q.groupby('subject').bal_acc.mean()
    def paired(a,b):
        a,b=subj_avg(a),subj_avg(b); j=a.index.intersection(b.index)
        if len(j)<5: return np.nan
        return stats.wilcoxon(a[j],b[j]).pvalue
    TABLES={}; lines=[]
    STATUS=json.load(open('rev05_status.json')) if os.path.exists('rev05_status.json') else {}
    R,P=load_all()   # all parts, merged
    R.to_csv(f'{OUT}/runs.csv',index=False); pickle.dump(P,open(f'{OUT}/preds.pkl','wb'))
    def show(name,df):
        TABLES[name]=df; df.to_csv(f'{OUT}/{name}.csv'); print(f'\n=== {name}'); print(df.round(2).to_string())

    # dataset statistics (Reviewer 2)
    st=[]
    for s in RAW:
        st.append(dict(subject=s['name'],train_trials=s['xtr'].shape[2],valid_trials=s['xte'].shape[2],
                       train_per_class=str(np.bincount(s['ytr'],minlength=5).tolist()),valid_per_class=str(np.bincount(s['yte'],minlength=5).tolist()),
                       samples=s['xtr'].shape[0],seconds=s['xtr'].shape[0]/FS,channels=s['xtr'].shape[1],
                       train_class_sorted=bool(np.all(np.diff(s['ytr'])>=0)),valid_class_sorted=bool(np.all(np.diff(s['yte'])>=0)),
                       nan=bool(np.isnan(s['xtr']).any() or np.isnan(s['xte']).any())))
    show('dataset_stats',pd.DataFrame(st).set_index('subject'))
    W=window_indices(RAW[0]['xtr'].shape[0]); print('analysis windows (s):',[(round(a/FS,3),round(b/FS,3)) for a,b in W])
    # epoch definition stored in the files (time axis relative to the cue, if present), for the methods text
    _md=_check(scipy.io.loadmat(train_files[0],struct_as_record=False,squeeze_me=True))
    _epo=next(_md[k] for k in ['epo_train','epo_test','epo_validation','epo'] if k in _md)
    print('mat keys:',[k for k in _md if not k.startswith('__')],'| epo fields:',list(_epo.keys()))
    for f in ['t','fs','className','clab']:
        if f in _epo:
            v=np.atleast_1d(_epo[f]); print(f'epo.{f}:', (v[0],v[-1],len(v)) if f=='t' else v[:64])

    # E1: variance over runs
    e1={reg:summ(subj_mean('E1',reg)) for reg in ['sd','pooled','si'] if len(subj_mean('E1',reg))}
    if e1:
        show('E1_regimes_over_runs',pd.DataFrame(e1).T)
        piv=R[R.exp=='E1'].groupby(['subject','regime']).bal_acc.mean().unstack()
        show('E1_per_subject_mean_over_runs',piv)
        if {'pooled','si'}<=set(piv.columns): print('pooled > SI in',int((piv.pooled>piv.si).sum()),'of',len(piv),'subjects; p =',paired(subj_mean('E1','pooled'),subj_mean('E1','si')))
        if {'sd','pooled'}<=set(piv.columns): print('SD vs pooled p =',paired(subj_mean('E1','sd'),subj_mean('E1','pooled')))
        if 'si' in piv.columns: print('SI vs chance (one-sided Wilcoxon) p =',stats.wilcoxon(piv.si-20,alternative='greater').pvalue)

    # E3: equal-size pooled vs SI
    if len(subj_mean('E3','pooled_eq')):
        eq,si=subj_mean('E3','pooled_eq'),subj_mean('E1','si')
        show('E3_pooled_equal_size',pd.DataFrame({'pooled_eq (14 subj incl. target)':summ(eq),'SI (14 other subj)':summ(si),
                                                  'pooled (15 subj)':summ(subj_mean('E1','pooled'))}).T)
        print('pooled_eq - SI gap:',round(subj_avg(eq).mean()-subj_avg(si).mean(),2),'pp; Wilcoxon p =',paired(eq,si))

    # E6: permutation null
    for reg in ['sd','pooled']:
        q=subj_mean('E6',reg)
        if len(q):
            null=per_run(q); obs=per_run(subj_mean('E1',reg)).mean() if len(subj_mean('E1',reg)) else np.nan
            show(f'E6_null_{reg}',pd.DataFrame(dict(n_perm=[len(null)],null_mean=[null.mean()],null_sd=[null.std(ddof=1)],
                  null_max=[null.max()],observed=[obs],p_perm=[(1+(null>=obs).sum())/(1+len(null))])))

    # E4: SD cross-validation under three fold constructions
    e4={sch:summ(subj_mean('E4',f'sd_cv_{sch}')) for sch in ['shuffled','blocked','group'] if len(subj_mean('E4',f'sd_cv_{sch}'))}
    if e4:
        show('E4_sd_cv_schemes',pd.DataFrame(e4).T)
        show('E4_per_subject',R[R.exp=='E4'].pivot_table(index='subject',columns='regime',values='bal_acc'))
        for a,b in [('shuffled','blocked'),('shuffled','group'),('blocked','group')]:
            if a in e4 and b in e4: print(f'{a} vs {b}: p =',paired(subj_mean('E4',f'sd_cv_{a}'),subj_mean('E4',f'sd_cv_{b}')))

    # E5
    if os.path.exists(f'{OUT}/E5_drift_controls.csv'):
        E5=pd.read_csv(f'{OUT}/E5_drift_controls.csv').set_index('subject'); show('E5_drift_controls',E5); print(E5.mean(numeric_only=True).round(1).to_string())

    # E7: artefact controls
    rows={}
    for v in ['base','central','hp1','hp1_central']:
        exp='E1' if v=='base' else 'E7'
        for reg in ['sd','pooled']:
            q=subj_mean(exp,reg,variant=v); q=q[q.seed.isin([0,1,2])]
            if len(q): rows[(v,reg)]=summ(q)
        q=subj_mean('E4' if v=='base' else 'E7','sd_cv_group',variant=v)
        if len(q): rows[(v,'sd_cv_group')]=summ(q)
    if len(rows)>3: show('E7_artefact_controls',pd.DataFrame(rows).T)

    # E2: capacity and alignment
    rows={}
    for v,mw in [('base','w25k'),('base','w100k'),('base','w330k'),('ea','w25k'),('ea','w330k')]:
        for reg in ['pooled','si']:
            exp='E1' if (v,mw)==('base','w25k') else 'E2'
            q=subj_mean(exp,reg,variant=v,model=mw)
            if reg=='pooled': q=q[q.seed.isin([0,1])]
            else: q=q[q.seed==0]
            if len(q): rows[(v,mw,reg)]=summ(q)
    if len(rows)>2: show('E2_capacity_alignment',pd.DataFrame(rows).T)

    # E8: EEGNet under the three regimes
    e8={reg:summ(subj_mean('E8',reg,model='eegnet')) for reg in ['sd','pooled','si'] if len(subj_mean('E8',reg,model='eegnet'))}
    if e8: show('E8_eegnet_regimes',pd.DataFrame(e8).T)

    json.dump(dict(status=STATUS,hours=(time.time()-T0)/3600,n_runs=len(R)),open(f'{OUT}/status.json','w'),indent=1)
    with zipfile.ZipFile(f'{RESULTS_DIR}/revision_results.zip','w',zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(f'{OUT}/*'):
            if not os.path.basename(f).startswith('maps_'): z.write(f,os.path.relpath(f,RESULTS_DIR))
    print('\nwrote',f'{RESULTS_DIR}/revision_results.zip','| status:',STATUS)


def stats_part():
    for k,w in WIDTHS.items(): print(k,f'{build_model(**w).count_params():,} params'); keras.backend.clear_session()
    if PAPER_MAPS:   # the rebuilt base maps must equal the paper's cache
        chk=make_maps(RAW[0]['xte']); print('base maps identical to paper cache (S01 valid):',np.allclose(chk,base[0]['Xte'],atol=1e-6))

if __name__=='__main__':
    import threading
    part=sys.argv[1]; t0=time.time()
    if part!='summary':   # watchdog: never run past the deadline (+5 min), whatever the code is doing
        wd=threading.Timer(max(1.0,DEADLINE+300-time.time()),lambda: os._exit(3)); wd.daemon=True; wd.start()
    try:
        if part=='stats': stats_part()
        elif part=='E5': run_e5()
        elif part=='summary': summary()
        else: PARTS[part]()
    except OutOfTime:
        print('time budget reached during',part); sys.exit(3)
    except Restart:
        print(f'{part}: {N_FITS} models trained in this process; continuing in a fresh one'); sys.exit(4)
    print(f'{part}: complete [{(time.time()-t0)/60:.1f} min]')

## Run the selected parts (one process each)

In [ ]:
import threading, queue
T0=time.time(); DEADLINE=T0+SETTINGS['TIME_BUDGET_H']*3600; STATUS={}
NOISE=('I0000','W0000','E0000','WARNING: All log messages','oneDNN','cuda_','computation placer','TF-TRT','Unable to register')
try: N_GPU=len(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout.strip().splitlines())
except Exception: N_GPU=0
WORKERS=int(os.environ.get('REV05_WORKERS',max(1,N_GPU)))   # one part per GPU at a time (Kaggle "GPU T4 x2" -> 2 parts in parallel)
LOCK=threading.Lock()
def run_part(part,gpu=None):
    """Run one part to completion, restarting it in a fresh process while it exits with code 4."""
    env=dict(os.environ, REV05_DEADLINE=str(DEADLINE), PYTHONUNBUFFERED='1', TF_CPP_MIN_LOG_LEVEL='2')
    if gpu is not None: env['CUDA_VISIBLE_DEVICES']=str(gpu)
    tag=f'[{part}] ' if gpu is None else f'[{part} gpu{gpu}] '
    while True:
        p=subprocess.Popen([sys.executable,'revision_experiments.py',part],stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT,text=True,env=env)
        for line in p.stdout:
            if not line.lstrip().startswith(NOISE):
                with LOCK: print(tag+line,end='',flush=True)
        rc=p.wait()
        if rc!=4 or time.time()>DEADLINE: return rc
def worker(i,q):
    gpu=i if N_GPU>0 else None
    while True:
        try: part=q.get_nowait()
        except queue.Empty: return
        if time.time() > DEADLINE-900: STATUS[part]='not started (time budget)'; continue
        rc=run_part(part,gpu)
        STATUS[part]={0:'complete',3:'stopped (time budget)',4:'stopped (time budget)'}.get(rc,f'failed (exit code {rc})')
        with LOCK:
            print(f'>>> {part}: {STATUS[part]}   [elapsed {(time.time()-T0)/3600:.2f} h]',flush=True)
            json.dump(STATUS,open('rev05_status.json','w'),indent=1)
# 'stats' and 'E5' first (fast), then the GPU parts, one per GPU at a time
for part in [p for p in RUN if p in ('stats','E5')]:
    rc=run_part(part); STATUS[part]='complete' if rc==0 else f'failed (exit code {rc})'
    print(f'>>> {part}: {STATUS[part]}',flush=True)
q=queue.Queue()
for part in RUN:
    if part not in ('stats','E5'): q.put(part)
threads=[threading.Thread(target=worker,args=(i,q)) for i in range(WORKERS)]
for th in threads: th.start()
for th in threads: th.join()
json.dump(STATUS,open('rev05_status.json','w'),indent=1)
print(STATUS)

## Summary tables

Means are over the 15 subjects; for repeated runs, the subject mean is computed per run and the table gives the
mean ± standard deviation of that quantity across runs. Paired tests across subjects are Wilcoxon signed-rank.
Send `revision_results.zip` (in the output folder) back for the manuscript.

In [ ]:
rc=run_part('summary'); print('summary exit code',rc)